<a href="https://colab.research.google.com/github/veronicaluzzi/DS-Capstone/blob/main/Repo/medicare_data_minimal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medicare data



From [Doctors and clinicians datasets](https://data.cms.gov/provider-data/search?theme=Doctors%20and%20clinicians) page
in the [National Downloadable File](https://data.cms.gov/provider-data/search?theme=Doctors%20and%20clinicians#:~:text=Doctors%20and%20clinicians-,National%20Downloadable%20File,-The%20Doctors%20and) section.


## DuckDB: query a remote CSV file



In [ ]:
import duckdb


In [ ]:
url = "https://data.cms.gov/provider-data/sites/default/files/resources/52c3f098d7e56028a298fd297cb0b38d_1774905035/DAC_NationalDownloadableFile.csv"
url


'https://data.cms.gov/provider-data/sites/default/files/resources/52c3f098d7e56028a298fd297cb0b38d_1774905035/DAC_NationalDownloadableFile.csv'

In [ ]:
!curl -s -I {url}


HTTP/2 200 
accept-ranges: bytes
content-security-policy: frame-ancestors https://*.cms.gov; upgrade-insecure-requests;
content-type: text/csv
last-modified: Mon, 30 Mar 2026 21:11:00 GMT
x-age: 0
x-ah-environment: prod
x-content-type-options: nosniff
x-request-id: v-00a20bb4-33e0-11f1-9b79-e3a98d70e175
content-length: 711273527
cache-control: max-age=81843
expires: Thu, 14 May 2026 01:49:39 GMT
date: Wed, 13 May 2026 03:05:36 GMT
strict-transport-security: max-age=31536000 ; includeSubDomains ; preload
x-xss-protection: 1; mode=block
akamai-grn: 0.550ed217.1778641536.82a51d9



## DuckDB: query a remote CSV file with a CTE


In [ ]:
duckdb.query(f"""
  SET force_download=true;

  WITH
  medicare as (
    SELECT *
    FROM read_csv_auto("{url}")
  ),

  medicare_nm as (
    SELECT *
    FROM medicare
    WHERE State = 'NM'
  ),

  medicare_abq as (
    SELECT *
    FROM medicare_nm
    WHERE "City/Town" IN ['ALBUQERQUE', 'RIO RANCHO']
  )

  SELECT *
  FROM medicare_abq
  ORDER BY "City/Town", NPI
  LIMIT 10
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬─────────────────┬────────────────────┬─────────────────────┬──────────────────────┬─────────┬─────────┬─────────┬─────────────────────────────────────────────────────┬────────┬───────────────────────────────────────────────┬────────────┬────────────┬────────────┬────────────┬──────────────┬──────────┬──────────────────────────────────────────────────┬────────────┬─────────────┬────────────────────────┬─────────────────────────────────────┬───────────┬────────────┬─────────┬───────────┬──────────────────┬───────────┬───────────┬───────────────────────────┐
│    NPI     │ Ind_PAC_ID │   Ind_enrl_ID   │ Provider Last Name │ Provider First Name │ Provider Middle Name │  suff   │  gndr   │  Cred   │                       Med_sch                       │ Grd_yr │                   pri_spec                    │ sec_spec_1 │ sec_spec_2 │ sec_spec_3 │ sec_spec_4 │ sec_spec_all │ Telehlth │                  Facility Name                   │ org_pac_id │ num_org_mem │  

In [ ]:
# brings the file to my vm,
df = duckdb.query(f"""
  SET force_download=true;

  WITH
  medicare as (
    SELECT *
    FROM read_csv("{url}", all_varchar = True)
  ),

  medicare_nm as (
    SELECT *
    FROM medicare
    WHERE State = 'NM'
  ),

  medicare_abq as (
    SELECT *
    FROM medicare_nm
    WHERE "City/Town" IN ['ALBUQERQUE', 'RIO RANCHO']
  )

  SELECT *
  FROM medicare
  -- ORDER BY "City/Town", NPI
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
df.shape


(2857460, 31)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2857460 entries, 0 to 2857459
Data columns (total 31 columns):
 #   Column                Dtype 
---  ------                ----- 
 0   NPI                   object
 1   Ind_PAC_ID            object
 2   Ind_enrl_ID           object
 3   Provider Last Name    object
 4   Provider First Name   object
 5   Provider Middle Name  object
 6   suff                  object
 7   gndr                  object
 8   Cred                  object
 9   Med_sch               object
 10  Grd_yr                object
 11  pri_spec              object
 12  sec_spec_1            object
 13  sec_spec_2            object
 14  sec_spec_3            object
 15  sec_spec_4            object
 16  sec_spec_all          object
 17  Telehlth              object
 18  Facility Name         object
 19  org_pac_id            object
 20  num_org_mem           object
 21  adr_ln_1              object
 22  adr_ln_2              object
 23  ln_2_sprs             object
 24

In [ ]:
df.head()


,NPI,Ind_PAC_ID,Ind_enrl_ID,Provider Last Name,Provider First Name,Provider Middle Name,suff,gndr,Cred,Med_sch,...,adr_ln_1,adr_ln_2,ln_2_sprs,City/Town,State,ZIP Code,Telephone Number,ind_assgn,grp_assgn,adrs_id
0,1245084383,9436636826,I20260115003111,LOPEZ CONCEPCION,RYAN,None,None,M,MD,OTHER,...,None,None,None,AGUADA,PR,00602,9393499867,Y,M,PR00602XXXXAGXXXXXXXXXX00
1,1568251114,1153817721,I20260102002504,FULGENCIO,TOSCANIA,Y.,None,F,MD,OTHER,...,None,None,None,AGUADA,PR,00602,7875890003,Y,M,PR00602XXXXAGXXXXXXXXXX00
2,1376594978,4587658521,I20260226002111,EISENBERG,DANNY,None,None,M,MD,OTHER,...,None,None,None,DORADO,PR,00646,2817420469,Y,Y,PR00646XXXXDOXXXXXXXXXX00
3,1801462064,4183117484,I20260210000126,JORGE CORIANO,MARIO,None,None,M,MD,OTHER,...,None,None,None,LAS MARIAS,PR,00670,7873653783,Y,M,PR00670XXXXLAXXXXXXXXXX00
4,1518522515,7416336912,I20251118002855,COLON VARGAS,JAVIER,EMANUEL,None,M,MD,PONCE SCHOOL OF MEDICINE,...,None,None,None,MOCA,PR,00676,7873964209,Y,M,PR00676XXXXMOXXXXXXXXXX00


In [ ]:
df.to_parquet('medicare.parquet')

In [1]:
duckdb.query(f'''
  SELECT *
  FROM read_parquet('https://drive.google.com/file/d/1tqv5m5fgcej57_zpEsQNyvGlZK3B2Kk3/view?usp=sharing')
  LIMIT 10
  ;
''').df()

NameError: name 'duckdb' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
